In [30]:
import folium
m = folium.Map(location=[35.68159659061569, 139.76451516151428], zoom_start=16)
from folium.plugins import TimestampedGeoJson

# Lon, Lat order.
lines = [
    {
        "coordinates": [
            [139.76451516151428, 35.68159659061569],
            [139.75964426994324, 35.682590062684206],
        ],
        "dates": ["2017-06-02T00:00:00", "2017-06-02T00:10:00"],
        "color": "red",
    },
    {
        "coordinates": [
            [139.75964426994324, 35.682590062684206],
            [139.7575843334198, 35.679505030038506],
        ],
        "dates": ["2017-06-02T00:10:00", "2017-06-02T00:20:00"],
        "color": "blue",
    },
    {
        "coordinates": [
            [139.7575843334198, 35.679505030038506],
            [139.76337790489197, 35.678040905014065],
        ],
        "dates": ["2017-06-02T00:20:00", "2017-06-02T00:30:00"],
        "color": "green",
        "weight": 15,
    },
    {
        "coordinates": [
            [139.76337790489197, 35.678040905014065],
            [139.76451516151428, 35.68159659061569],
        ],
        "dates": ["2017-06-02T00:30:00", "2017-06-02T00:40:00"],
        "color": "#FFFFFF",
    },
]

features = [
    {
        "type": "Feature",
        "geometry": {
            "type": "LineString",
            "coordinates": line["coordinates"],
        },
        "properties": {
            "times": line["dates"],
            "style": {
                "color": line["color"],
                "weight": line["weight"] if "weight" in line else 5,
            },
        },
    }
    for line in lines
]

TimestampedGeoJson(
    {
        "type": "FeatureCollection",
        "features": features,
    },
    period="PT1M",
    duration="PT1M",
    add_last_point=True,
    speed_slider=False,
).add_to(m)

m

In [35]:
# 104.0143061429087 30.6295471457352
from envs.crowd_sim.utils import get_border
from datasets.Chengdu.data_preprocess import get_longitude_and_latitude_max
import folium
lower_left = [104.04215, 30.65294]
upper_right = list(get_longitude_and_latitude_max(lower_left[0], lower_left[1], 6000))
my_render_map: folium.Map = folium.Map(location=[lower_left[1], lower_left[0]], tiles="cartodbpositron", zoom_start=14, max_zoom=24, control_scale=True)
grid_geo_json = get_border(upper_right, lower_left)
color = 'red'
border = folium.GeoJson(grid_geo_json,
                        style_function=lambda feature, clr=color: {
                            # 'fillColor': color,
                            'color': "black",
                            'weight': 2,
                            'dashArray': '5,5',
                            'fillOpacity': 0,
                        }).add_to(my_render_map)
my_render_map

In [48]:
# open all csv starting with prefix emergency_time_loc
import os
import pandas as pd
parent_path = os.path.join('/workspace/Awesome-Mobile-Crowdsensing', 'datasets/Chengdu')
file_names = os.listdir(parent_path)
file_names = [file_name for file_name in file_names if file_name.startswith('emergency_time_loc')]
file_names = sorted(file_names)
common = set()
for file_name in file_names:
    print(file_name)
    df = pd.read_csv(os.path.join(parent_path, file_name))
    # print unique number counts
    print("unique number counts")
    print(df.groupby(['x_bin', 'y_bin']).size().count())
    # print len
    print("len")
    print(len(df))
    # check how many common (x_bin, y_bin) pairs in all csv
    if len(common) == 0:
        common = set(df.groupby(['x_bin', 'y_bin']).size().index)
    else:
        common = common.intersection(set(df.groupby(['x_bin', 'y_bin']).size().index))
print(common)